## Databricks notebook source
## SCB Bronze Layer: Workforce Data
Ingests raw JSON from the SCB PxWebApi v2 into an external volume, partitioned by ingestion date.
**API**: `https://api.scb.se/OV0104/v1/doris/en/ssd` (PxWebApi v2)
**Target**: `/Volumes/labour_market_platform/bronze_work_scb/landing/scb/` (external volume on ADLS)
| Subfolder | Output File(s) | SCB Endpoint | Description |
|---|---|---|---|
| `wages/` | `wages_2016_2022.json`, `wages_2023_2024.json` | `AM/AM0110/AM0110A/LoneSpridSektorYrk4A`, `LoneSpridSektYrk4AN` | Wage distribution by sector, SSYK 2012 occupation |
| `aku/` | `aku_employment_stock_occupation.json` | `AM/AM0401/AM0401I/NAKUSysselYrke2012Ar` | Employment stock by occupation (AKU) |
| `aku/` | `aku_population_region_labourstatus.json` | `AM/AM0401/AM0401N/NAKUBefolkningLAr` | Population by region and labour status (AKU) |
| `aku/` | `aku_unemployed_age_sex.json` | `AM/AM0401/AM0401L/NAKUArblheltidstudAr` | Unemployment by age and sex (AKU) |
| `reference/` | `ssyk_2012_mapping.json` | `AM/AM0110/AM0110A/LoneSpridSektorYrk4A` (Swedish) | SSYK 2012 occupation code metadata |
**Ingestion strategy**:
* Bulk POST for all years per endpoint; falls back to per-year fetch on non-2xx
* Rate-limit handling with incremental backoff on 429
* Partitioned by `ingestion_date=YYYY-MM-DD` for full history
## Credentials & Imports

In [0]:
# Imports
import json
import os
import time
import requests
from datetime import datetime, timezone

# Credentials 
CATALOG      = "labour_market_platform"
SCHEMA       = "bronze_work_scb"
VOLUME       = "landing"

SCB_BASE_URL = "https://api.scb.se/OV0104/v1/doris/en/ssd"
BRONZE_ROOT  = "scb"
BASE_PATH    = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

#validates values
print(f"Ingestion date: {INGESTION_DATE}")
print(f"Base path:      {BASE_PATH}/{BRONZE_ROOT}/")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS labour_market_platform.bronze_work_scb;

CREATE EXTERNAL VOLUME IF NOT EXISTS labour_market_platform.bronze_work_scb.landing
  LOCATION 'abfss://landing@adlmlabourmarket.dfs.core.windows.net/raw';

In [0]:
def fetch_scb_metadata(endpoint_path: str) -> dict:
    """Fetches metadata to discover available years and variables."""
    response = requests.get(f"{SCB_BASE_URL}/{endpoint_path}")
    response.raise_for_status()
    return response.json()
 
 
def build_scb_query(metadata: dict, years: list) -> dict:
    """Builds POST query body, selecting all variables for the specified years."""
    query_items = []
    for variable in metadata["variables"]:
        code = variable["code"]
        if code == "Tid":
            query_items.append({
                "code": "Tid",
                "selection": {"filter": "item", "values": years},
            })
        else:
            query_items.append({
                "code": code,
                "selection": {"filter": "item", "values": variable["values"]},
            })
    return {"query": query_items, "response": {"format": "json"}}
 
 
def post_scb_query(endpoint_path: str, query_body: dict) -> dict:
    """Posts query to SCB API and returns raw JSON."""
    response = requests.post(f"{SCB_BASE_URL}/{endpoint_path}", json=query_body)
    response.raise_for_status()
    return response.json()

In [0]:
def write_json_to_bronze(raw_json: dict, subfolder: str, filename: str) -> None:
    """Writes JSON to external volume, partitioned by date."""
    dir_path  = f"{BASE_PATH}/{BRONZE_ROOT}/{subfolder}/ingestion_date={INGESTION_DATE}"
    file_path = f"{dir_path}/{filename}"

    os.makedirs(dir_path, exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(raw_json, f, ensure_ascii=False)

In [0]:
def ingest_scb_dataset(endpoint_path: str, years: list, subfolder: str, filename: str) -> None:
    """Fetches data from SCB. Tries bulk request first, falls back to per-year on failure."""
    metadata        = fetch_scb_metadata(endpoint_path)
    available_years = next(v["values"] for v in metadata["variables"] if v["code"] == "Tid")
    years_to_fetch  = [y for y in years if y in available_years]
 
    if not years_to_fetch:
        print(f"  No matching years. Available: {available_years}")
        return
 
    print(f"  Years: {years_to_fetch}")
 
    # Attempt bulk fetch
    try:
        query    = build_scb_query(metadata, years_to_fetch)
        raw_json = post_scb_query(endpoint_path, query)
        write_json_to_bronze(raw_json, subfolder, filename)
        time.sleep(2)
        return
    except requests.exceptions.HTTPError:
        pass # Fallback triggered

    # Fallback: Per-year fetch with rate limit backoff
    merged_data = []
    columns     = None

    for year in years_to_fetch:
        for attempt in range(3):
            try:
                time.sleep(3)
                query    = build_scb_query(metadata, [year])
                raw_json = post_scb_query(endpoint_path, query)

                merged_data.extend(raw_json.get("data", []))
                if columns is None:
                    columns = raw_json.get("columns")
                break
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 429:
                    time.sleep(15 * (attempt + 1))
                else:
                    break

    if merged_data:
        write_json_to_bronze({"columns": columns, "data": merged_data}, subfolder, filename)
        print(f"  Total rows: {len(merged_data)}")

In [0]:
# Wages split across two SCB table versions (API schema change at 2023)
ingest_scb_dataset(
    "AM/AM0110/AM0110A/LoneSpridSektorYrk4A",
    [str(y) for y in range(2016, 2023)],
    "wages",
    "wages_2016_2022.json",
)

ingest_scb_dataset(
    "AM/AM0110/AM0110A/LoneSpridSektYrk4AN",
    ["2023", "2024"],
    "wages",
    "wages_2023_2024.json",
)

In [0]:
# AKU (Arbetskraftsundersökningen) — employment, population, and unemployment
aku_years = [str(y) for y in range(2016, 2026)]

ingest_scb_dataset(
    "AM/AM0401/AM0401I/NAKUSysselYrke2012Ar",
    aku_years,
    "aku",
    "aku_employment_stock_occupation.json",
)

ingest_scb_dataset(
    "AM/AM0401/AM0401N/NAKUBefolkningLAr",
    aku_years,
    "aku",
    "aku_population_region_labourstatus.json",
)

ingest_scb_dataset(
    "AM/AM0401/AM0401L/NAKUArblheltidstudAr",
    aku_years,
    "aku",
    "aku_unemployed_age_sex.json",
)

In [0]:
# SSYK 2012 occupation code metadata (Swedish endpoint for Swedish occupation names)
ssyk_metadata_url = "https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/AM0110/AM0110A/LoneSpridSektorYrk4A"

print("\n--- ssyk_2012_mapping.json ---")

response = requests.get(ssyk_metadata_url)
response.raise_for_status()
ssyk_metadata = response.json()

write_json_to_bronze(ssyk_metadata, "reference", "ssyk_2012_mapping.json")

In [0]:
print("\n--- Bronze ingestion complete ---")
print(f"Path: {BASE_PATH}/{BRONZE_ROOT}/")
print(f"Partition: ingestion_date={INGESTION_DATE}")